# Keras Sequential vs Functional vs Subclassing: When to Use Which API

> This notebook contains all code examples from the blog post.
> [Read the full post on BotMartz](https://blog.botmartz.com/tensorflow_2)

**Author:** Soham Sharma · blog.botmartz.com

In [ ]:
!pip install -q tensorflow mlflow

In [ ]:
import tensorflow as tf

model = tf.keras.Sequential([
    tf.keras.layers.Dense(128, activation='relu', input_shape=(20,)),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(10, activation='softmax'),
])

model.summary()

In [ ]:
import tensorflow as tf

# Define inputs
inputs = tf.keras.Input(shape=(20,), name="features")

# Build the graph
x = tf.keras.layers.Dense(128, activation='relu')(inputs)
x = tf.keras.layers.Dropout(0.3)(x)
x = tf.keras.layers.Dense(64, activation='relu')(x)

# Skip connection: add inputs (projected) to intermediate output
shortcut = tf.keras.layers.Dense(64)(inputs)
x = tf.keras.layers.Add()([x, shortcut])
x = tf.keras.layers.Activation('relu')(x)

output = tf.keras.layers.Dense(10, activation='softmax', name="predictions")(x)

# Model is defined by its inputs and outputs
model = tf.keras.Model(inputs=inputs, outputs=output, name="residual_classifier")
model.summary()

In [ ]:
import tensorflow as tf

# Two separate input streams
text_input = tf.keras.Input(shape=(100,), name="text_features")
meta_input = tf.keras.Input(shape=(10,), name="metadata")

# Process each stream
text_encoded = tf.keras.layers.Dense(64, activation='relu')(text_input)
meta_encoded = tf.keras.layers.Dense(16, activation='relu')(meta_input)

# Merge streams
merged = tf.keras.layers.Concatenate()([text_encoded, meta_encoded])
merged = tf.keras.layers.Dense(32, activation='relu')(merged)

# Two output heads
sentiment = tf.keras.layers.Dense(1, activation='sigmoid', name="sentiment")(merged)
topic = tf.keras.layers.Dense(5, activation='softmax', name="topic")(merged)

model = tf.keras.Model(
    inputs={"text_features": text_input, "metadata": meta_input},
    outputs={"sentiment": sentiment, "topic": topic},
    name="multi_task_classifier"
)

print(f"Inputs: {list(model.input_names)}")
print(f"Outputs: {list(model.output_names)}")

In [ ]:
import tensorflow as tf
import numpy as np

# Rebuild the multi-task model from above
text_input = tf.keras.Input(shape=(100,), name="text_features")
meta_input = tf.keras.Input(shape=(10,), name="metadata")
text_enc = tf.keras.layers.Dense(64, activation='relu')(text_input)
meta_enc = tf.keras.layers.Dense(16, activation='relu')(meta_input)
merged = tf.keras.layers.Concatenate()([text_enc, meta_enc])
merged = tf.keras.layers.Dense(32, activation='relu')(merged)
sentiment = tf.keras.layers.Dense(1, activation='sigmoid', name="sentiment")(merged)
topic = tf.keras.layers.Dense(5, activation='softmax', name="topic")(merged)

model = tf.keras.Model(
    inputs={"text_features": text_input, "metadata": meta_input},
    outputs={"sentiment": sentiment, "topic": topic},
)

model.compile(
    optimizer='adam',
    loss={"sentiment": "binary_crossentropy", "topic": "sparse_categorical_crossentropy"},
    loss_weights={"sentiment": 1.0, "topic": 0.5},
    metrics={"sentiment": "accuracy", "topic": "accuracy"},
)

# Dummy data
N = 200
x_data = {"text_features": np.random.randn(N, 100), "metadata": np.random.randn(N, 10)}
y_data = {"sentiment": np.random.randint(0, 2, N), "topic": np.random.randint(0, 5, N)}

history = model.fit(x_data, y_data, epochs=2, batch_size=32, verbose=1)

In [ ]:
import tensorflow as tf

class ResidualBlock(tf.keras.layers.Layer):
    """A single residual block."""
    def __init__(self, units: int, **kwargs):
        super().__init__(**kwargs)
        self.dense1 = tf.keras.layers.Dense(units, activation='relu')
        self.dense2 = tf.keras.layers.Dense(units)
        self.projection = tf.keras.layers.Dense(units)
        self.add = tf.keras.layers.Add()
        self.relu = tf.keras.layers.Activation('relu')

    def call(self, inputs, training=False):
        x = self.dense1(inputs)
        x = self.dense2(x)
        shortcut = self.projection(inputs)
        x = self.add([x, shortcut])
        return self.relu(x)

class ResidualClassifier(tf.keras.Model):
    def __init__(self, num_classes: int, **kwargs):
        super().__init__(**kwargs)
        self.block1 = ResidualBlock(64)
        self.block2 = ResidualBlock(32)
        self.output_layer = tf.keras.layers.Dense(num_classes, activation='softmax')

    def call(self, inputs, training=False):
        x = self.block1(inputs, training=training)
        x = self.block2(x, training=training)
        return self.output_layer(x)

model = ResidualClassifier(num_classes=10)

# Build by passing dummy input
dummy = tf.zeros([1, 20])
_ = model(dummy)
model.summary()

In [ ]:
import tensorflow as tf

class ClassifierWithDropout(tf.keras.Model):
    def __init__(self):
        super().__init__()
        self.dense = tf.keras.layers.Dense(64, activation='relu')
        self.dropout = tf.keras.layers.Dropout(0.5)
        self.out = tf.keras.layers.Dense(2, activation='softmax')

    def call(self, inputs, training=False):
        x = self.dense(inputs)
        x = self.dropout(x, training=training)  # ← must pass training here
        return self.out(x)

model = ClassifierWithDropout()
x = tf.random.normal([4, 10])

# training=True: dropout is active
train_out = model(x, training=True)
# training=False: dropout is disabled
infer_out = model(x, training=False)

print(f"Train output sum: {train_out.numpy().sum():.4f}")
print(f"Infer output sum: {infer_out.numpy().sum():.4f}")

In [ ]:
import tensorflow as tf
import tempfile, os

# Functional model — serializes perfectly
inputs = tf.keras.Input(shape=(10,))
outputs = tf.keras.layers.Dense(5, activation='relu')(inputs)
functional_model = tf.keras.Model(inputs, outputs)

with tempfile.TemporaryDirectory() as tmpdir:
    path = os.path.join(tmpdir, "model")
    functional_model.save(path)
    loaded = tf.keras.models.load_model(path)
    result = loaded(tf.ones([2, 10]))
    print(f"Loaded functional model output shape: {result.shape}")

In [ ]:
import tensorflow as tf

class MultiHeadAttentionBlock(tf.keras.layers.Layer):
    def __init__(self, num_heads: int, key_dim: int, **kwargs):
        super().__init__(**kwargs)
        self.attention = tf.keras.layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=key_dim
        )
        self.norm = tf.keras.layers.LayerNormalization()
        self.ffn = tf.keras.layers.Dense(key_dim * num_heads, activation='relu')

    def call(self, inputs, training=False):
        attn_output = self.attention(inputs, inputs)
        x = self.norm(inputs + attn_output)
        return self.ffn(x)

# Use the custom layer in a Functional model
seq_input = tf.keras.Input(shape=(32, 64), name="sequence")  # (batch, seq_len, dim)
x = MultiHeadAttentionBlock(num_heads=4, key_dim=16)(seq_input)
x = tf.keras.layers.GlobalAveragePooling1D()(x)
output = tf.keras.layers.Dense(3, activation='softmax')(x)

model = tf.keras.Model(inputs=seq_input, outputs=output)
print(f"Input: {model.input_shape}, Output: {model.output_shape}")